In [1]:
import pandas as pd

df = pd.read_csv("../data/quant/SRR3501467/abundance.tsv", sep="\t")
df.head()

,target_id,length,eff_length,est_counts,tpm
0,ENST00000415118.1,8,2.48058,0.0,0.0
1,ENST00000448914.1,13,2.62202,0.0,0.0
2,ENST00000434970.2,9,2.52391,0.0,0.0
3,ENST00000631435.1,12,2.60424,0.0,0.0
4,ENST00000632684.1,12,2.60424,0.0,0.0


In [2]:
print(df.shape)
print(df["tpm"].sum())
df.sort_values("tpm", ascending=False).head(10)

(206723, 5)
999999.9623735456


,target_id,length,eff_length,est_counts,tpm
133009,ENST00000501597.3,469,270.000,52231.2,7621.35
112559,ENST00000229239.10,1285,1086.000,152259.0,5523.56
626,ENST00000362079.2,784,585.000,79020.3,5321.68
622,ENST00000361624.2,1542,1343.000,165102.0,4843.31
61644,ENST00000646664.1,1812,1613.000,184838.0,4514.64
194806,ENST00000510231.1,177,10.528,1106.0,4138.80
154157,ENST00000392514.9,1105,906.000,83270.3,3620.99
82385,ENST00000336023.9,1627,1428.000,129153.0,3563.22
59712,ENST00000676710.1,1790,1591.000,142558.0,3530.10
110087,ENST00000651669.1,352,153.000,13304.6,3425.91


In [3]:
import gzip, re
import pandas as pd

# A "regular expression": a pattern that picks fields out of each header line.
pattern = re.compile(
    r"^>(\S+) \S+ \S+:GRCh38:([^:]+):(\d+):(\d+):(-?1) "
    r"gene:(\S+) gene_biotype:(\S+) transcript_biotype:(\S+)"
    r"(?: gene_symbol:(\S+))?"
)

rows = []
unmatched = 0
with gzip.open("../data/ref/Homo_sapiens.GRCh38.cdna.all.fa.gz", "rt") as f:
    for line in f:
        if line.startswith(">"):          # header lines only, skip the sequences
            m = pattern.match(line)
            if m:
                rows.append(m.groups())
            else:
                unmatched += 1

tx = pd.DataFrame(rows, columns=[
    "transcript_id", "chrom", "start", "end", "strand",
    "gene_id", "gene_biotype", "transcript_biotype", "gene_symbol",
])
tx["start"] = tx["start"].astype(int)
tx["end"] = tx["end"].astype(int)

print("parsed:", len(tx), " unmatched headers:", unmatched)
tx.head()

parsed: 206723  unmatched headers: 0


,transcript_id,chrom,start,end,strand,gene_id,gene_biotype,transcript_biotype,gene_symbol
0,ENST00000415118.1,14,22438547,22438554,1,ENSG00000223997.1,TR_D_gene,TR_D_gene,TRDD1
1,ENST00000448914.1,14,22449113,22449125,1,ENSG00000228985.1,TR_D_gene,TR_D_gene,TRDD3
2,ENST00000434970.2,14,22439007,22439015,1,ENSG00000237235.2,TR_D_gene,TR_D_gene,TRDD2
3,ENST00000631435.1,HSCHR7_2_CTG6,809186,809197,1,ENSG00000282253.1,TR_D_gene,TR_D_gene,TRBD1
4,ENST00000632684.1,7,142786213,142786224,1,ENSG00000282431.1,TR_D_gene,TR_D_gene,TRBD1


In [4]:
keep = [str(i) for i in range(1, 23)] + ["X", "Y"]
on_main = tx["chrom"].isin(keep)

print("transcripts total:", len(tx))
print("on chr1-22, X, Y:", on_main.sum())
print(tx.loc[~on_main, "chrom"].value_counts().head(8))

transcripts total: 206723
on chr1-22, X, Y: 187043
chrom
HSCHR6_MHC_COX_CTG1     1164
HSCHR6_MHC_QBL_CTG1     1131
HSCHR6_MHC_DBB_CTG1     1122
HSCHR6_MHC_MCF_CTG1     1093
HSCHR6_MHC_SSTO_CTG1    1015
HSCHR6_MHC_MANN_CTG1     944
HSCHR6_MHC_APD_CTG1      540
HG1_PATCH                435
Name: count, dtype: int64


In [5]:
top = df.sort_values("tpm", ascending=False).head(10)
top.merge(tx, left_on="target_id", right_on="transcript_id")[
    ["target_id", "tpm", "gene_symbol", "chrom", "gene_biotype"]
]

,target_id,tpm,gene_symbol,chrom,gene_biotype
0,ENST00000501597.3,7621.35,RPL41,12,protein_coding
1,ENST00000229239.10,5523.56,GAPDH,12,protein_coding
2,ENST00000362079.2,5321.68,MT-CO3,MT,protein_coding
3,ENST00000361624.2,4843.31,MT-CO1,MT,protein_coding
4,ENST00000646664.1,4514.64,ACTB,7,protein_coding
5,ENST00000510231.1,4138.80,NaN,4,processed_pseudogene
6,ENST00000392514.9,3620.99,RPLP0,12,protein_coding
7,ENST00000336023.9,3563.22,TUBA1B,12,protein_coding
8,ENST00000676710.1,3530.10,EEF1A1,6,protein_coding
9,ENST00000651669.1,3425.91,RPS27,1,protein_coding


In [6]:
samples = {
    "CSES9": "SRR3501465",
    "CSES7": "SRR3501466",
    "CSES22": "SRR3501467",
    "CSES21": "SRR3501468",
    "HUES14": "SRR3501469",
}

# keep only protein-coding genes on chr1-22, X, Y
tx_keep = tx[tx["chrom"].isin(keep) & (tx["gene_biotype"] == "protein_coding")]

# one row per gene: chromosome, start, end, symbol
genes = tx_keep.groupby("gene_id").agg(
    chrom=("chrom", "first"),
    start=("start", "min"),
    end=("end", "max"),
    symbol=("gene_symbol", "first"),
    n_chrom=("chrom", "nunique"),
)
assert (genes["n_chrom"] == 1).all()   # every gene sits on exactly one chromosome

# sum transcript counts to gene counts, for each sample
counts = {}
for name, run in samples.items():
    q = pd.read_csv(f"../data/quant/{run}/abundance.tsv", sep="\t")
    total = q["est_counts"].sum()
    q = q.merge(tx_keep[["transcript_id", "gene_id"]],
                left_on="target_id", right_on="transcript_id")
    counts[name] = q.groupby("gene_id")["est_counts"].sum()
    print(f"{name}: kept {q['est_counts'].sum() / total:.1%} of estimated counts")

counts = pd.DataFrame(counts)
print(counts.shape, genes.shape)
counts.head()

CSES9: kept 92.6% of estimated counts
CSES7: kept 88.1% of estimated counts
CSES22: kept 94.8% of estimated counts
CSES21: kept 93.1% of estimated counts
HUES14: kept 92.5% of estimated counts
(20052, 5) (20052, 5)


,CSES9,CSES7,CSES22,CSES21,HUES14
gene_id,,,,,
ENSG00000000003.16,1976.252100,1308.06100,3618.477900,3401.60800,6069.40180
ENSG00000000005.6,23.999980,23.00003,82.000000,60.00000,53.00000
ENSG00000000419.14,860.084440,829.32311,2051.096390,1857.68653,2020.51496
ENSG00000000457.14,179.829500,173.82372,309.963600,429.74470,296.38271
ENSG00000000460.17,1291.144265,671.18947,1021.484352,1321.86970,1749.06110


In [7]:
cpm = counts / counts.sum() * 1e6     # counts per million, to compare across samples
markers = ["POU5F1", "NANOG", "LIN28A", "SOX2", "DNMT3B",   # pluripotency
           "SOX17", "PAX6", "GATA6", "TBXT"]                 # differentiation
idx = genes.index[genes["symbol"].isin(markers)]
out = cpm.reindex(idx)
out.insert(0, "symbol", genes.loc[idx, "symbol"])
out.round(1).sort_values("symbol")

,symbol,CSES9,CSES7,CSES22,CSES21,HUES14
gene_id,,,,,,
ENSG00000088305.19,DNMT3B,3176.2,1823.8,893.1,612.0,2947.7
ENSG00000141448.11,GATA6,0.3,45.6,6.3,13.2,0.9
ENSG00000131914.11,LIN28A,1927.7,1489.6,2164.5,1360.0,1463.2
ENSG00000111704.11,NANOG,140.0,112.3,21.8,75.9,155.2
ENSG00000007372.25,PAX6,0.2,0.9,0.2,2.2,0.3
ENSG00000204531.21,POU5F1,1.4,1131.8,4.6,1.5,1.9
ENSG00000164736.6,SOX17,0.0,33.2,2.6,6.8,0.4
ENSG00000181449.4,SOX2,217.5,217.6,200.5,147.5,253.0
ENSG00000164458.10,TBXT,0.2,6.5,0.9,7.4,0.3


In [9]:
import os
os.makedirs("../data/processed", exist_ok=True)
counts.to_csv("../data/processed/gene_counts.csv")
genes.to_csv("../data/processed/genes.csv")

In [10]:
sym = "POU5F1"
rows = []
for name, run in samples.items():
    q = pd.read_csv(f"../data/quant/{run}/abundance.tsv", sep="\t")
    q = q.merge(tx[["transcript_id", "chrom", "gene_symbol"]],
                left_on="target_id", right_on="transcript_id")
    sub = q[q["gene_symbol"] == sym]
    rows.append(sub.groupby("chrom")["est_counts"].sum().rename(name))
pd.concat(rows, axis=1).fillna(0).round(0)

,CSES9,CSES7,CSES22,CSES21,HUES14
chrom,,,,,
6,35.0,19418.0,165.0,66.0,64.0
HSCHR6_MHC_COX_CTG1,19189.0,7.0,8.0,17050.0,32615.0
HSCHR6_MHC_DBB_CTG1,19189.0,7.0,8.0,17050.0,32615.0
HSCHR6_MHC_MANN_CTG1,46.0,0.0,3.0,22.0,24.0
HSCHR6_MHC_MCF_CTG1,46.0,0.0,3.0,22.0,24.0
HSCHR6_MHC_QBL_CTG1,10.0,35.0,13896.0,3.0,7.0
HSCHR6_MHC_SSTO_CTG1,10.0,35.0,13896.0,3.0,7.0
